In [13]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from urllib.parse import quote_plus

if os.path.exists("data"):
    DATA_DIR = "data"
elif os.path.exists("../data"):
    DATA_DIR = "../data"
else:
    raise FileNotFoundError("❌ Папка data не найдена!")

print(f"✅ Папка данных: {DATA_DIR}")

✅ Папка данных: ../data


In [14]:
DB_USER = "postgres"
DB_PASSWORD = "1199"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "neobank"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)
print("✅ Подключение установлено")

✅ Подключение установлено


In [15]:
users = pd.read_sql('SELECT * FROM "User";', engine)
cards = pd.read_sql('SELECT * FROM "Card";', engine)
transactions = pd.read_sql('SELECT * FROM "Transaction";', engine)
credits = pd.read_sql('SELECT * FROM "Credit";', engine)

print(f"👥 Пользователей: {len(users)}")
print(f"💳 Карт: {len(cards)}")
print(f"📊 Транзакций: {len(transactions)}")
print(f"🏦 Кредитов: {len(credits)}")

👥 Пользователей: 10
💳 Карт: 20
📊 Транзакций: 284
🏦 Кредитов: 8


In [16]:
transactions['date'] = pd.to_datetime(transactions['date'])

tx_with_user = transactions.merge(
    cards[['id', 'userId']],
    left_on='cardId',
    right_on='id',
    suffixes=('', '_card')
)

print(f"Транзакций после merge: {len(tx_with_user)}")
print(f"Диапазон дат: {transactions['date'].min()} — {transactions['date'].max()}")

Транзакций после merge: 284
Диапазон дат: 2025-12-22 00:00:00 — 2026-10-03 14:23:21.582000


In [17]:
now = transactions['date'].max()
three_months_ago = now - pd.DateOffset(months=3)

print(f"📅 Дата отсчёта: {now}")
print(f"📅 Окно: {three_months_ago.date()} — {now.date()}")

tx_recent = tx_with_user[tx_with_user['date'] >= three_months_ago]
print(f"📊 Транзакций в окне: {len(tx_recent)} из {len(tx_with_user)}")

📅 Дата отсчёта: 2026-10-03 14:23:21.582000
📅 Окно: 2026-07-03 — 2026-10-03
📊 Транзакций в окне: 67 из 284


In [18]:
features = []
for uid in users['id']:
    user_tx = tx_recent[tx_recent['userId'] == uid]
    
    income = user_tx[user_tx['type'] == 'plus']['amount'].sum() / 3
    expense = user_tx[user_tx['type'] == 'minus']['amount'].sum() / 3
    debt_ratio = expense / income if income > 0 else 1.0
    
    user_data = users[users['id'] == uid].iloc[0]
    user_cards = cards[(cards['userId'] == uid) & (cards['isBlocked'] == False)]
    user_credits = credits[credits['userId'] == uid]
    
    days_on_platform = (now - pd.to_datetime(user_data['createdAt'])).days
    closed_credits = (user_credits['status'] == 'closed').sum() if len(user_credits) > 0 else 0
    has_overdue = 1 if len(user_credits) > 0 and (
        (user_credits['penalty'] > 0).any() or (user_credits['status'] == 'overdue').any()
    ) else 0
    
    features.append({
        'userId': uid,
        'income': income,
        'expense': expense,
        'debt_ratio': debt_ratio,
        'balance': user_data['balance'],
        'active_cards': len(user_cards),
        'days_on_platform': days_on_platform,
        'closed_credits': closed_credits,
        'has_overdue': has_overdue,
    })

df_features = pd.DataFrame(features)
print(f"✅ Собрано признаков: {df_features.shape}")
df_features

✅ Собрано признаков: (10, 9)


,userId,income,expense,debt_ratio,balance,active_cards,days_on_platform,closed_credits,has_overdue
0,9,35000.000000,17066.666667,0.487619,178550.000000,2,225,0,0
1,10,20000.000000,2766.666667,0.138333,225400.000000,1,209,0,0
2,1,21666.666667,8832.817011,0.407668,145140.548968,5,375,4,0
3,2,21666.666667,6333.333333,0.292308,175300.000000,2,345,0,0
4,3,35000.000000,12333.333333,0.352381,146650.000000,3,315,1,0
5,4,15000.000000,2150.000000,0.143333,210250.000000,1,295,0,0
6,5,40000.000000,16000.000000,0.400000,183800.000000,1,285,0,0
7,6,18333.333333,2300.000000,0.125455,177000.000000,1,270,0,0
8,7,30000.000000,5066.666667,0.168889,229742.100000,2,255,1,0
9,8,16666.666667,3616.666667,0.217000,135050.000000,1,240,0,0


In [19]:
df_features.to_csv(f"{DATA_DIR}/features.csv", index=False)
print(f"✅ Файл {DATA_DIR}/features.csv создан")

✅ Файл ../data/features.csv создан
